# Read quantiles, refusals and variance effects

**Question:** How do quantile denominators and finite-design Sobol estimates affect uncertainty interpretation?

Level: 2. Prerequisites: earlier lessons in this topic. Reading time: 30 minutes.



## Objectives

The practical uncertainty calculation separates solved-output quantiles from all-attempt feasibility, then examines first/total Sobol effects at two finite budgets. Every summary is computed through the production engine at the declared seed.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Import the pinned analysis tools once and resolve inputs relative to the notebook folder.

In [ ]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Data

Owner: OreFlow authors, MIT project licence. The production case catalog defines the nominal soft-porphyry ore and circuit. engine/data/constants.json defines the independent uniform ore-factor half-widths, seed-related method defaults and sensitivity tolerances. The notebook passes its stated sample counts and seed 20261031 to the production uncertainty methods. No operating-contract file, deployed screen, archived study table or measured uncertainty distribution is loaded here. Source: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

The practical lesson first calculates the same 32-draw conditional output distribution, retaining attempted and solved denominators. It then calls the production Saltelli/Sobol sensitivity method at base N=8 and N=16 with the same seed and factor ranges. For four factors these designs require 48 and 96 circuit evaluations respectively. Structured hybrid rows cannot be dropped after a refusal; the method reports that invalid design instead. First-order and total effects answer different variance questions, and their bootstrap half-widths quantify finite-estimator stability. Constant output is treated as a declared numerical condition rather than ranked from roundoff.

Fix the nominal operating point while examining two different uncertainty summaries. Propagation estimates a conditional output distribution, whereas Sobol analysis decomposes its variance among independent factors. Both use the same authored ore ranges and exact circuit engine. They do not use the same sampling matrix or answer the same question. Start with nominal recovery so the later distribution and variance contributions have a common physical reference.

In [ ]:
# Start with the authored soft-porphyry circuit and its nominal ore.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.methods.uncertainty import uncertainty, sensitivity, perturbed

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
print("Nominal recovery (%):", round(nominal.metrics["recovery_pct"], 3))


Compute a lower liberation-scale response to expose how one factor enters the physical model. The perturbation changes mineral accessibility, not the final output directly. This deterministic control is useful when a later total-effect index appears large: it helps distinguish a plausible physical pathway from a feature that was merely given a wide input range. It cannot estimate a global variance share by itself.

In [ ]:
# Resolve the lower liberation-size factor of 0.9 while holding other ore factors at unity.
low_ore, low_point = perturbed(case, case.nominal, {"liberation_size": 0.9})
low = simulate(low_ore, case.plant, low_point)
print("Liberation factor:", 0.9)
print("Conditional recovery (%):", round(low.metrics["recovery_pct"], 3))


Compute the upper liberation-scale response independently. Keeping all other factors at unity makes these two controls locally interpretable. Their difference from nominal is neither a confidence interval nor a sensitivity derivative averaged over the entire domain. Interactions with head grade, work index and floatability appear only when those factors also vary. The comparison therefore motivates a global design rather than replacing it.

In [ ]:
# Resolve the upper liberation-size factor of 1.1 while holding other ore factors at unity.
high_ore, high_point = perturbed(case, case.nominal, {"liberation_size": 1.1})
high = simulate(high_ore, case.plant, high_point)
print("Liberation factor:", 1.1)
print("Conditional recovery (%):", round(high.metrics["recovery_pct"], 3))


Propagate the 32-draw Latin hypercube and preserve both output and constraint summaries. P05 and P95 are empirical quantiles of solved states. The constraint probability includes rejected draws as failures, so it may deteriorate even when the conditional recovery interval looks harmless. Report the actual solved denominator and refusal reasons whenever a downstream reader might compare intervals between operating points.

In [ ]:
# Propagate 32 Latin-hypercube draws through the production engine using a fixed seed.
experiment = uncertainty(case, case.nominal, samples=32, seed=20261031)
print("Solved draws:", len(experiment["solved"]))
print("Refused draws:", experiment["refused"])
print("Factor half-widths:", experiment["inputs"])
summary = pd.DataFrame(experiment["outputs"]).T[["p05", "p50", "p95", "std"]]
display(summary.round(4))
print("All-constraint probability:", round(experiment["probabilities"]["all_constraints"], 4))


Show the solved recovery distribution before reducing it to moments. A standard deviation and a P05-P95 interval retain different aspects of shape. The histogram can reveal asymmetry caused by nonlinear response or limits, though 32 draws cannot support detailed tail modeling. Its nominal marker is an engine reference only and does not supply an observed expected recovery.

In [ ]:
# Plot the recovery draws and nominal reference so the sampled distribution is visible rather than
# reduced to three quantiles..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
recovery = np.asarray(experiment["outputs"]["recovery_pct"]["values"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.hist(recovery, bins=8, edgecolor="white")
ax.axvline(nominal.metrics["recovery_pct"], color="darkred", label="Nominal")
ax.set(
    xlabel="Recovery (%)",
    ylabel="Solved draws\n(count)",
    title="Ore-factor uncertainty propagation",
)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)


Read the shape as conditional on these independent uniform factors. A plant distribution would require geological dependence and measurement calibration, neither supplied by stratification. A concentration near one edge can arise from model constraints rather than evidence for an ore regime. Retain the factor definitions with this figure, because widening one range can change the apparent importance of every other factor through the total output variance.

Run the first Saltelli design with eight base rows. Each factor's hybrid rows enable separation of first-order contribution and total contribution including interactions. Every row must be physically solved; dropping a refused row would break the estimator's matrix structure. The production method therefore refuses to report indices when a draw is unsolved. Keep that refusal as a scientific result instead of patching the design with a nominal output.

In [ ]:
# Calculate the Saltelli/Sobol variance decomposition at base N=8.
small_indices = sensitivity(case, case.nominal, base_samples=8, seed=20261031)
print("Sobol base rows:", small_indices["base_samples"])
print("Sobol engine evaluations:", small_indices["evaluations"])
print("Sobol status:", small_indices["status"])
if small_indices["status"] != "computed":
    raise RuntimeError("Sensitivity design contains refused draws")
effects = small_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("Recovery is constant at the declared tolerance")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Double the base design to sixteen using the same seed and factor ranges. This changes the sampling budget without changing the physical model. Finite estimates can move substantially at these deliberately small sizes, and a small negative first-order value is sampling error rather than a negative physical variance. Bootstrap half-widths quantify this estimator variability, not uncertainty about the true operating plant.

In [ ]:
# Calculate the Saltelli/Sobol variance decomposition at base N=16.
larger_indices = sensitivity(case, case.nominal, base_samples=16, seed=20261031)
print("Sobol base rows:", larger_indices["base_samples"])
print("Sobol engine evaluations:", larger_indices["evaluations"])
print("Sobol status:", larger_indices["status"])
if larger_indices["status"] != "computed":
    raise RuntimeError("Sensitivity design contains refused draws")
effects = larger_indices["indices"]["recovery_pct"]
if effects.get("constant"):
    raise RuntimeError("Recovery is constant at the declared tolerance")
display(pd.DataFrame({key: effects[key] for key in ("S1", "ST", "ST_conf")}))


Plot total-order indices from both budgets with their reported bootstrap half-widths. Shared factor names and unitless vertical scale make shifts directly comparable. Do not normalize the bars to sum to one: total effects include interactions and their sum need not equal unity. The design's exact evaluation count is printed separately so the cost of a more stable estimate remains visible.

In [ ]:
# Compare total-order sensitivity at the two sample budgets, including the production bootstrap
# half-widths..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
names = list(larger_indices["inputs"])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for offset, record in ((-0.1, small_indices), (0.1, larger_indices)):
    effect = record["indices"]["recovery_pct"]
    values = [effect["ST"][name] for name in names]
    widths = [effect["ST_conf"][name] for name in names]
    ax.errorbar(
        values,
        np.arange(len(names)) + offset,
        xerr=widths,
        fmt="o",
        label="N=" + str(record["base_samples"]),
    )
ax.set_yticks(range(len(names)), [name.replace("_", " ") for name in names], rotation=0)
ax.set(xlabel="Total Sobol\nindex (1)", ylabel="", title="Finite-design sensitivity stability")
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


Overlapping wide intervals prevent a confident ranking even if the point estimates appear ordered. A consistent large total effect suggests a useful factor to examine further under these assumptions, not an independently established causal driver. This lesson combines distribution interpretation, refusal handling and finite-design stability. A decision study needs larger repeated designs and defensible joint ore distributions before quoting a robust risk or importance ordering.

A useful consistency check is to distinguish the number of attempted factor rows from the number of solved recoveries. If every draw solves, both denominators coincide and the distinction is easy to miss. If a state is refused, conditioning the entire constraint calculation on successful solves would make the operating point appear more reliable than the attempted scenario design supports. The production propagator instead preserves refusal counts and evaluates joint feasibility over attempts. Do not multiply marginal constraint probabilities to reconstruct joint feasibility: grade, power and water are coupled through the same draws.

## Results

**Answer:** The output interval <!-- result:Recovery P05 (%) -->0<!-- /result --> to <!-- result:Recovery P95 (%) -->0<!-- /result --> percent uses <!-- result:Solved count -->0<!-- /result --> solved recoveries, while feasibility <!-- result:Joint feasibility (1) -->0<!-- /result --> includes all attempts and <!-- result:Refused count -->0<!-- /result --> refusals. The larger Sobol design still has maximum total-effect half-width <!-- result:Largest total-effect half-width (1) -->0<!-- /result -->, so a neat point-estimate ordering alone cannot certify sensitivity stability.

Collect propagation denominators and the larger Sobol design half-width for the finite-budget interpretation.

In [ ]:
# Summarize the computed outcomes used in this lesson's interpretation.
answer_values = {
    "Recovery P05 (%)": experiment["outputs"]["recovery_pct"]["p05"],
    "Recovery P95 (%)": experiment["outputs"]["recovery_pct"]["p95"],
    "Solved count": len(experiment["solved"]),
    "Refused count": sum(experiment["refused"].values()),
    "Joint feasibility (1)": experiment["probabilities"]["all_constraints"],
}
order_small = sorted(
    small_indices["inputs"], key=small_indices["indices"]["recovery_pct"]["ST"].get
)
order_large = sorted(
    larger_indices["inputs"], key=larger_indices["indices"]["recovery_pct"]["ST"].get
)
answer_values.update(
    {
        "Same total-effect order (1=yes)": int(order_small == order_large),
        "Largest total-effect half-width (1)": max(
            larger_indices["indices"]["recovery_pct"]["ST_conf"].values()
        ),
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Summary

Solved-quantile and all-attempt feasibility denominators answer different questions. The two finite sensitivity budgets add estimator variability and interaction context; their intervals belong beside the point estimates before any factor ranking is interpreted.

## References

- https://doi.org/10.1080/00401706.1979.10489755